# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-32: App-level battery - v3 against v2

### What this notebook does
Decides, under a rule fixed before any battery result existed (Part 0), whether `hr_physnet_v3` replaces `hr_physnet_v2` as the app's default heart-rate estimator. Both models were scored through the app's own estimator code and quality gates by four benchmark scripts at a single commit. This notebook verifies the provenance of those outputs, evaluates every criterion and states the outcome; it runs no model, reads no video and trains nothing.

### Models
- `hr_physnet_v2` - the shipped model: NB_P3_18, configuration `baseline`, epoch 27.
- `hr_physnet_v3` - NB_P3_31 arm A (v2's training corpora plus VitalVideos-WW), seed 42, last epoch with BatchNorm statistics re-estimated.
- `hr_spectral` (POS) - unsupervised reference, reported on every set and never decisive.

### Benchmark sets
| Set | Recordings | Role | Benchmark script |
|:---|---:|:---|:---|
| VitalVideos-WW validation split (VV-60) | 60 | primary (Fitzpatrick 5-6); guard | `check_vitalvideos` |
| UBFC-Phys, task T1 | 56 | guard | `check_ubfc_phys` |
| v2's validation split: MCD, DLCN, UBFC-rPPG | 407 | guard | `check_phase3_stores` |
| PhysDrive, all indexed recordings | 266 | stress test | `check_phase3_stores` |
| ECG-Fitness (NB_P3_29 Part 0 primary set: 80) | 100 | stress test | `check_ecg_fitness` |

### Background
NB_P3_31 found arm A's per-window error at Fitzpatrick 5-6 well below v2's (recording level, A - v2 -3.11 bpm, 95% CI -4.77 to -1.46). The app does not report windows: it reports a reading or withholds one. The battery tests whether that gain reaches users as reported readings, and whether anything else degrades.

## 0. Decision rule (Part 0)

*Pre-registered in the commit the battery ran on, before the first battery run. This cell restates that rule in its final layout; no definition, threshold or outcome differs from the committed text.*

**Comparison.** `hr_physnet_v3` against `hr_physnet_v2`, both evaluated through the app's estimator code and quality gates. `hr_spectral` (POS) is evaluated alongside as a reference and does not enter the decision.

**Definitions** (per recording unless stated otherwise)

| Term | Definition |
|:---|:---|
| eligible | the recording's reference yields a reading (`ref_accepted`); on ECG-Fitness, the primary set of NB_P3_29 Part 0 |
| reported | app status `ok`, `degraded_capture` or `unstable` |
| error | reported value minus the reference heart rate (`ref_hr`; on ECG-Fitness `ref_hr_median`, the median of the recording's ECG-referenced windows) |
| wrong | reported, and more than 10 bpm from the reference (the threshold of NB_P3_29 Part 0) |
| gain / loss | an eligible recording reported only by v3 / only by v2; net = gains - losses |
| like-for-like window MAE | window MAE on the primary-set windows both models read (`common_windows` in each summary) |

**Criteria**

| # | Kind | Set | Met when |
|:---|:---|:---|:---|
| 1 | Primary | VV-60, Fitzpatrick 5-6 (23 recordings) | all three hold: net ≥ +2 with at most 1 loss; v3 has no more wrong readings than v2; on the recordings both report, v3's mean absolute error is at most 1.0 bpm above v2's |
| 2 | Guard | VV-60, Fitzpatrick 1-4 (37 recordings), and all 60 | net ≥ -1, and no more wrong readings than v2 |
| 2 | Guard | UBFC-Phys T1 (56 recordings, held out from both models) | net ≥ -2, and like-for-like window MAE at most +0.5 bpm |
| 2 | Guard | v2's validation split of MCD, DLCN and UBFC-rPPG (407 recordings) | like-for-like window MAE at most +0.5 bpm pooled (`shared`) and at most +0.35 bpm on MCD (NB_P3_31's guard margins); pooled coverage (reported / eligible) at most 2 percentage points lower |
| 3 | Stress | PhysDrive (all 266 indexed recordings; no model trained on PhysDrive) and ECG-Fitness Part 0's primary set (80 recordings) | v3 has at most 2 more wrong readings than v2 |
| 4 | Drift | UBFC-rPPG regression checks for v2, v3 and POS | all three PASS on the battery's code and weights, before the battery runs |

**Outcomes**
- Criteria 1, 2 and 3 met: v3 becomes `DEFAULT_ESTIMATOR`; v2 remains registered.
- Criterion 1 not met, criteria 2 and 3 met: v2 remains the default; v3 remains a registered candidate.
- Any guard or stress test not met: v2 remains the default, and the cause is established before further v3 work.

**Scope.** The outcome is a decision about the app's default model, made on 23 recordings at Fitzpatrick 5-6. It is not evidence that v3 serves darker skin better: with no real coverage difference the coverage criterion would still pass about 10-17% of the time; a properly powered test of a 12-point coverage gain needs about 75 recordings at types 5-6; and VV-60 already selected arm A over R in NB_P3_31, so this reading leans optimistic. A claim about skin tone requires an independent set.

**Reported as description, not decisive.** McNemar's exact one-sided p on the gains and losses; subject-bootstrap 95% intervals (4,000 draws, seed 42) for the VV-60 coverage and error differences; counts by Fitzpatrick type and site group, as aggregates only (no recording is shown with its type or site); agreement with VitalVideos' ECG; the window-level metrics of ECG-Fitness Part 0; POS on every set.

## 1. Inputs and provenance
Loads each model's summary and recording table for the four sets and verifies that all of them come from one commit with no uncommitted app code, use the registered weights, are complete runs, cover the expected recordings, and score the same recordings for every model. The cell stops if any check fails.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest


def find_repo_root(start):
    '''The first folder at or above `start` holding both app/ and Data/.'''
    for folder in [start, *start.parents]:
        if (folder / 'app').is_dir() and (folder / 'Data').is_dir():
            return folder
    raise FileNotFoundError(f'no repository root at or above {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA = REPO_ROOT / 'Data'
V2, V3, POS = 'hr_physnet_v2', 'hr_physnet_v3', 'hr_spectral'
MODELS = (V2, V3, POS)
WEIGHTS = {V2: 'f30fae250fb948e2aaece9aae5ec2d77fcc438d7ef1abbe54ae878a7eecee6b8',   # NB_P3_18, epoch 27
           V3: '3fab82a89f4863a70c18ddb3d8f8f7cbc0d923adf65f85b2d0aeaa56895e0600',   # NB_P3_31 A, seed 42, last_bn
           POS: None}
# Each battery set: output folder and the name suffix its run writes.
RUNS = {'VV-60': ('vitalvideos_eval', '_split-val'),
        'UBFC-Phys T1': ('ubfc_phys_eval', '_tasks-T1'),
        'Phase-3 held-out': ('phase3_eval', '_split-heldout'),
        'ECG-Fitness': ('ecg_fitness_eval', '')}
EXPECTED = {'VV-60': 60, 'UBFC-Phys T1': 56, 'Phase-3 held-out': 673, 'ECG-Fitness': 100,
            'shared': 407, 'PhysDrive': 266, 'ECG-Fitness primary': 80}
REPORTED = ('ok', 'degraded_capture', 'unstable')
WRONG_BPM = 10.0
SHARED = ('MCD', 'DLCN', 'UBFC-rPPG')


def load(run, model):
    '''One model's summary and recording rows for one battery set.'''
    folder, suffix = RUNS[run]
    summary = json.loads((DATA / folder / f'summary_{model}{suffix}.json').read_text(encoding='utf-8'))
    recordings = pd.read_csv(DATA / folder / f'recordings_{model}{suffix}.csv')
    return summary, recordings


OUT = {(run, m): load(run, m) for run in RUNS for m in MODELS}

problems, commits = [], set()
for (run, m), (summary, recs) in OUT.items():
    prov = summary.get('provenance') or {}
    weights = (prov.get('weights') or {}).get('sha256')
    if weights != WEIGHTS[m]:
        problems.append(f'{run} {m}: weights {weights}, expected {WEIGHTS[m]}')
    git = prov.get('git') or {}
    commits.add(git.get('commit'))
    app_changes = [f for f in git.get('changed', []) if f.startswith('app/')]
    if app_changes:
        problems.append(f'{run} {m}: run on uncommitted app code {app_changes}')
    if summary.get('limit') is not None:
        problems.append(f'{run} {m}: a --limit run')
    if len(recs) != EXPECTED[run]:
        problems.append(f'{run} {m}: {len(recs)} recordings, expected {EXPECTED[run]}')
if len(commits) != 1:
    problems.append(f'the runs come from more than one commit: {sorted(map(str, commits))}')
for run in RUNS:
    first = list(OUT[(run, V2)][1]['recording'])
    if any(list(OUT[(run, m)][1]['recording']) != first for m in MODELS):
        problems.append(f'{run}: the models did not score the same recordings')
vv_summary = OUT[('VV-60', V2)][0]
if vv_summary['split'] != 'val' or vv_summary['speed'] != 1.0:
    problems.append(f"VV-60: split {vv_summary['split']}, speed {vv_summary['speed']}")
if OUT[('UBFC-Phys T1', V2)][0]['tasks_run'] != ['T1']:
    problems.append(f"UBFC-Phys: tasks {OUT[('UBFC-Phys T1', V2)][0]['tasks_run']}")
p3 = OUT[('Phase-3 held-out', V2)][0]
for group in ('shared', 'PhysDrive'):
    if p3['split'] != 'heldout' or p3['n_recordings'].get(group) != EXPECTED[group]:
        problems.append(f"Phase-3: split {p3['split']}, {group} {p3['n_recordings'].get(group)}")
n_ecg_primary = int(OUT[('ECG-Fitness', V2)][1]['primary'].sum())
if n_ecg_primary != EXPECTED['ECG-Fitness primary']:
    problems.append(f'ECG-Fitness: {n_ecg_primary} primary recordings')

print('commit:', ', '.join(map(str, commits)))
for run, (folder, suffix) in RUNS.items():
    print(f'  {run:17} Data/{folder}/summary_<model>{suffix}.json  {len(OUT[(run, V2)][1])} recordings')
assert not problems, 'preflight failed:\n  ' + '\n  '.join(problems)
print('provenance verified: one commit, registered weights, complete runs, expected recordings')

commit: 25ba5beb00a6c2b9ddebb08dcfb8299c89a89b8e
  VV-60             Data/vitalvideos_eval/summary_<model>_split-val.json  60 recordings
  UBFC-Phys T1      Data/ubfc_phys_eval/summary_<model>_tasks-T1.json  56 recordings
  Phase-3 held-out  Data/phase3_eval/summary_<model>_split-heldout.json  673 recordings
  ECG-Fitness       Data/ecg_fitness_eval/summary_<model>.json  100 recordings
provenance verified: one commit, registered weights, complete runs, expected recordings


## 2. Criteria and verdict
Evaluates criteria 1-3 from the recording tables and the like-for-like window blocks, then assigns the outcome Part 0 specifies. Criterion 4 is recorded rather than computed: the three UBFC-rPPG regression checks were run at commit `25ba5be` before the battery and all passed, each reproducing its baseline exactly (mean MAE 1.239 bpm for v2, 1.240 for v3 and 1.351 for POS).

In [2]:
def compare(run, ref, eligible, where=None):
    '''v3 against v2, recording by recording, on the eligible recordings of one set.'''
    v2 = OUT[(run, V2)][1].set_index('recording')
    v3 = OUT[(run, V3)][1].set_index('recording')
    rows = v2[v2[eligible].astype(bool)]
    if where is not None:
        rows = rows[where(rows)]
    ids = rows.index
    side = {}
    for name, table in ((V2, v2), (V3, v3)):
        t = table.loc[ids]
        reported = t['status'].isin(REPORTED).to_numpy()
        error = (t['value'] - t[ref]).to_numpy(dtype=float)
        side[name] = (reported, error, reported & (np.abs(error) > WRONG_BPM))
    (r2, e2, w2), (r3, e3, w3) = side[V2], side[V3]
    both = r2 & r3
    gains, losses = int((r3 & ~r2).sum()), int((r2 & ~r3).sum())
    return dict(n=len(ids), reported_v2=int(r2.sum()), reported_v3=int(r3.sum()),
                gains=gains, losses=losses, net=gains - losses,
                wrong_v2=int(w2.sum()), wrong_v3=int(w3.sum()), both=int(both.sum()),
                mae_both_v2=float(np.abs(e2[both]).mean()) if both.any() else np.nan,
                mae_both_v3=float(np.abs(e3[both]).mean()) if both.any() else np.nan,
                rows=dict(r2=r2, r3=r3, e2=e2, e3=e3))


def window_gap(run, group=None):
    '''v3 minus v2 window MAE on the primary-set windows both read, and their count.'''
    block = OUT[(run, V3)][0]['common_windows']
    m = (block[group] if group else block)[V2]['primary']
    return m['this'].get('mae', np.nan) - m['other'].get('mae', np.nan), m['n_both']


def at_most(x, limit):
    return bool(np.isfinite(x) and x <= limit + 1e-12)


def coverage(c):
    return f"gains {c['gains']}, losses {c['losses']}, net {c['net']:+d} of {c['n']}"


def wrong(c):
    return f"v2 {c['wrong_v2']}, v3 {c['wrong_v3']}"


vv_56 = compare('VV-60', 'ref_hr', 'ref_accepted', lambda d: d['fitzpatrick'].isin([5, 6]))
vv_14 = compare('VV-60', 'ref_hr', 'ref_accepted', lambda d: d['fitzpatrick'].between(1, 4))
vv_all = compare('VV-60', 'ref_hr', 'ref_accepted')
t1 = compare('UBFC-Phys T1', 'ref_hr', 'ref_accepted')
shared = compare('Phase-3 held-out', 'ref_hr', 'ref_accepted', lambda d: d['dataset'].isin(SHARED))
physdrive = compare('Phase-3 held-out', 'ref_hr', 'ref_accepted', lambda d: d['dataset'] == 'PhysDrive')
ecg = compare('ECG-Fitness', 'ref_hr_median', 'primary')
gap_t1, n_t1 = window_gap('UBFC-Phys T1', 'T1')
gap_shared, n_shared = window_gap('Phase-3 held-out', 'shared')
gap_mcd, n_mcd = window_gap('Phase-3 held-out', 'MCD')
shared_change = (shared['reported_v3'] - shared['reported_v2']) / shared['n'] if shared['n'] else np.nan
err_gap_56 = vv_56['mae_both_v3'] - vv_56['mae_both_v2']

CRITERIA = [  # (part, criterion, observed, rule, met)
    ('1 primary', 'VV-60 types 5-6: coverage', coverage(vv_56), 'net >= +2, losses <= 1',
     vv_56['net'] >= 2 and vv_56['losses'] <= 1),
    ('1 primary', 'VV-60 types 5-6: wrong readings', wrong(vv_56), 'v3 <= v2',
     vv_56['wrong_v3'] <= vv_56['wrong_v2']),
    ('1 primary', 'VV-60 types 5-6: error where both report',
     f"{vv_56['both']} recordings, v2 {vv_56['mae_both_v2']:.2f}, v3 {vv_56['mae_both_v3']:.2f} bpm",
     'v3 - v2 <= +1.0 bpm', at_most(err_gap_56, 1.0)),
    ('2 guard', 'VV-60 types 1-4: coverage', coverage(vv_14), 'net >= -1', vv_14['net'] >= -1),
    ('2 guard', 'VV-60 types 1-4: wrong readings', wrong(vv_14), 'v3 <= v2',
     vv_14['wrong_v3'] <= vv_14['wrong_v2']),
    ('2 guard', 'VV-60 all: coverage', coverage(vv_all), 'net >= -1', vv_all['net'] >= -1),
    ('2 guard', 'VV-60 all: wrong readings', wrong(vv_all), 'v3 <= v2',
     vv_all['wrong_v3'] <= vv_all['wrong_v2']),
    ('2 guard', 'UBFC-Phys T1: coverage', coverage(t1), 'net >= -2', t1['net'] >= -2),
    ('2 guard', 'UBFC-Phys T1: like-for-like window MAE', f'v3 - v2 {gap_t1:+.2f} bpm on {n_t1} windows',
     '<= +0.5 bpm', at_most(gap_t1, 0.5)),
    ('2 guard', "v2's validation, shared: like-for-like window MAE",
     f'v3 - v2 {gap_shared:+.2f} bpm on {n_shared} windows', '<= +0.5 bpm', at_most(gap_shared, 0.5)),
    ('2 guard', "v2's validation, MCD: like-for-like window MAE",
     f'v3 - v2 {gap_mcd:+.2f} bpm on {n_mcd} windows', '<= +0.35 bpm', at_most(gap_mcd, 0.35)),
    ('2 guard', "v2's validation, shared: coverage",
     f"v2 {shared['reported_v2']}, v3 {shared['reported_v3']} of {shared['n']} ({100 * shared_change:+.1f} points)",
     '>= -2 points', at_most(-shared_change, 0.02)),
    ('3 stress', 'PhysDrive: wrong readings', wrong(physdrive) + f" of {physdrive['n']}", 'v3 <= v2 + 2',
     physdrive['wrong_v3'] <= physdrive['wrong_v2'] + 2),
    ('3 stress', 'ECG-Fitness primary: wrong readings', wrong(ecg) + f" of {ecg['n']}", 'v3 <= v2 + 2',
     ecg['wrong_v3'] <= ecg['wrong_v2'] + 2),
]

print(f"{'part':9} {'criterion':50} {'observed':58} {'rule':24} result")
for part, name, observed, rule, met in CRITERIA:
    print(f"{part:9} {name:50} {observed:58} {rule:24} {'met' if met else 'NOT MET'}")

met = {p: all(c[4] for c in CRITERIA if c[0] == p) for p in ('1 primary', '2 guard', '3 stress')}
if all(met.values()):
    verdict = 'v3 becomes DEFAULT_ESTIMATOR; v2 stays registered.'
elif met['2 guard'] and met['3 stress']:
    verdict = 'v2 stays the default; v3 stays a registered candidate.'
else:
    verdict = 'v2 stays the default; the cause of the failed guard or stress test is found before more v3 work.'
print('\n' + ', '.join(f"{p}: {'met' if ok else 'NOT MET'}" for p, ok in met.items()))
print('4 drift: met (recorded: the three regression checks passed before the battery ran)')
print(f'VERDICT: {verdict}')

part      criterion                                          observed                                                   rule                     result
1 primary VV-60 types 5-6: coverage                          gains 1, losses 0, net +1 of 23                            net >= +2, losses <= 1   NOT MET
1 primary VV-60 types 5-6: wrong readings                    v2 0, v3 0                                                 v3 <= v2                 met
1 primary VV-60 types 5-6: error where both report           20 recordings, v2 0.70, v3 0.42 bpm                        v3 - v2 <= +1.0 bpm      met
2 guard   VV-60 types 1-4: coverage                          gains 0, losses 0, net +0 of 37                            net >= -1                met
2 guard   VV-60 types 1-4: wrong readings                    v2 0, v3 0                                                 v3 <= v2                 met
2 guard   VV-60 all: coverage                                gains 1, losses 0, net +1 of 60       

## 3. Descriptive results
The statistics Part 0 lists as description; none of them enters the verdict. Counts by Fitzpatrick type and site group are aggregates, and no recording appears with its type or site.

In [3]:
rng = np.random.default_rng(42)
N_BOOT = 4000


def mcnemar(c):
    '''Exact one-sided McNemar p for v3 reporting more than v2: the gains among the discordant.'''
    d = c['gains'] + c['losses']
    return binomtest(c['gains'], d, 0.5, alternative='greater').pvalue if d else float('nan')


def boot_interval(values):
    finite = values[np.isfinite(values)]
    return np.percentile(finite, [2.5, 97.5]) if finite.size >= 0.5 * values.size else [np.nan, np.nan]


def vv_bootstrap(c):
    '''Subject-bootstrap 95% intervals (one recording per VitalVideos subject) for the
    coverage difference and the error difference where both report.'''
    r2, r3, e2, e3 = (c['rows'][k] for k in ('r2', 'r3', 'e2', 'e3'))
    n = len(r2)
    cover, err = np.empty(N_BOOT), np.empty(N_BOOT)
    for b in range(N_BOOT):
        i = rng.integers(0, n, n)
        cover[b] = (r3[i].sum() - r2[i].sum()) / n
        both = r2[i] & r3[i]
        err[b] = np.abs(e3[i][both]).mean() - np.abs(e2[i][both]).mean() if both.any() else np.nan
    return boot_interval(cover), boot_interval(err)


print('VV-60, coverage and error differences (v3 - v2):')
for label, c in (('types 5-6', vv_56), ('types 1-4', vv_14), ('all 60', vv_all)):
    (c_lo, c_hi), (e_lo, e_hi) = vv_bootstrap(c)
    print(f"  {label:9} reported v2 {c['reported_v2']}/{c['n']}, v3 {c['reported_v3']}/{c['n']} | "
          f"coverage {100 * c['net'] / c['n']:+.1f} points (95% {100 * c_lo:+.1f} to {100 * c_hi:+.1f}) | "
          f"McNemar one-sided p {mcnemar(c):.3f} | error where both report "
          f"{c['mae_both_v3'] - c['mae_both_v2']:+.2f} bpm (95% {e_lo:+.2f} to {e_hi:+.2f})")

v2_rows = OUT[('VV-60', V2)][1]
v3_rows = OUT[('VV-60', V3)][1].set_index('recording').loc[v2_rows['recording']].reset_index()
eligible = v2_rows['ref_accepted'].astype(bool)
for field in ('fitzpatrick', 'site_group'):
    print(f'\nVV-60 by {field} (eligible recordings; reported and wrong, v2 / v3):')
    for value in sorted(v2_rows.loc[eligible, field].unique(), key=str):
        mask = eligible & (v2_rows[field] == value)
        cells = []
        for rows in (v2_rows, v3_rows):
            reported = rows.loc[mask, 'status'].isin(REPORTED)
            off = (rows.loc[mask, 'value'] - rows.loc[mask, 'ref_hr']).abs() > WRONG_BPM
            cells.append((int(reported.sum()), int((reported & off).sum())))
        print(f'  {str(value):12} n {int(mask.sum()):3d} | reported {cells[0][0]:3d} / {cells[1][0]:3d} | '
              f'wrong {cells[0][1]:2d} / {cells[1][1]:2d}')

print("\nVV-60 against the study's ECG (recordings the audit accepted, reported by both):")
ecg_ok = eligible & v2_rows['ecg_ok'].astype(bool)
both = ecg_ok & v2_rows['status'].isin(REPORTED) & v3_rows['status'].isin(REPORTED)
for name, rows in ((V2, v2_rows), (V3, v3_rows)):
    gap = (rows.loc[both, 'value'] - rows.loc[both, 'hr_ecg']).abs()
    print(f'  {name}: mean |value - ECG| {gap.mean():.2f} bpm on {int(both.sum())} recordings')

print('\nVV-60 like for like, primary set:')
gap_vv, n_vv = window_gap('VV-60')
print(f'  v3 - v2 window MAE {gap_vv:+.2f} bpm on {n_vv} windows both read')

print('\nPOS (hr_spectral), for reference - eligible, reported, wrong:')
for run, ref, elig, where, label in (
        ('VV-60', 'ref_hr', 'ref_accepted', lambda d: d['fitzpatrick'].isin([5, 6]), 'VV-60 types 5-6'),
        ('VV-60', 'ref_hr', 'ref_accepted', None, 'VV-60 all'),
        ('UBFC-Phys T1', 'ref_hr', 'ref_accepted', None, 'UBFC-Phys T1'),
        ('Phase-3 held-out', 'ref_hr', 'ref_accepted', lambda d: d['dataset'].isin(SHARED), 'shared'),
        ('Phase-3 held-out', 'ref_hr', 'ref_accepted', lambda d: d['dataset'] == 'PhysDrive', 'PhysDrive'),
        ('ECG-Fitness', 'ref_hr_median', 'primary', None, 'ECG-Fitness primary')):
    rows = OUT[(run, POS)][1]
    mask = rows[elig].astype(bool) & (where(rows) if where else True)
    reported = rows.loc[mask, 'status'].isin(REPORTED)
    off = (rows.loc[mask, 'value'] - rows.loc[mask, ref]).abs() > WRONG_BPM
    print(f'  {label:20} n {int(mask.sum()):4d} | reported {int(reported.sum()):4d} | wrong {int((reported & off).sum()):3d}')

print("\nECG-Fitness Part 0, window level by activity (confident-wrong among scored windows):")
for activity in OUT[('ECG-Fitness', V2)][0]['primary_by_activity']:
    parts = []
    for name in (V2, V3):
        g = OUT[('ECG-Fitness', name)][0]['primary_by_activity'][activity]
        parts.append(f"{name} {g['n_confident_wrong']}/{g['n_scored']} reported {g['n_reported']}/{g['n_recordings']}")
    print(f'  {activity}: ' + ' | '.join(parts))

VV-60, coverage and error differences (v3 - v2):
  types 5-6 reported v2 20/23, v3 21/23 | coverage +4.3 points (95% +0.0 to +13.0) | McNemar one-sided p 0.500 | error where both report -0.28 bpm (95% -0.53 to -0.06)
  types 1-4 reported v2 37/37, v3 37/37 | coverage +0.0 points (95% +0.0 to +0.0) | McNemar one-sided p nan | error where both report -0.13 bpm (95% -0.23 to -0.04)
  all 60    reported v2 57/60, v3 58/60 | coverage +1.7 points (95% +0.0 to +5.0) | McNemar one-sided p 0.500 | error where both report -0.19 bpm (95% -0.30 to -0.08)

VV-60 by fitzpatrick (eligible recordings; reported and wrong, v2 / v3):
  1            n   2 | reported   2 /   2 | wrong  0 /  0
  2            n  16 | reported  16 /  16 | wrong  0 /  0
  3            n   4 | reported   4 /   4 | wrong  0 /  0
  4            n  15 | reported  15 /  15 | wrong  0 /  0
  5            n  12 | reported  11 /  12 | wrong  0 /  0
  6            n  11 | reported   9 /   9 | wrong  0 /  0

VV-60 by site_group (eligibl

## 4. Discussion and conclusions

**Outcome: v2 remains the app's default heart-rate estimator, and v3 remains a registered candidate.** Criterion 1 was not met. At Fitzpatrick 5-6, v3 reported one recording more than v2 (21 against 20 of 23; one gain, no loss), short of the net +2 that Part 0 requires. The other two primary conditions held, as did every guard and both stress tests, and the three regression checks passed at the battery commit.

**v3's gain reaches the app as accuracy rather than coverage.** v2 already reported 20 of the 23 recordings at types 5-6, so coverage could rise by three at most. Where both models report, v3's readings are closer to the reference: 0.42 against 0.70 bpm at types 5-6 (difference -0.28 bpm, 95% CI -0.53 to -0.06), -0.13 bpm at types 1-4 (-0.23 to -0.04) and -0.19 bpm across all 60 (-0.30 to -0.08). Against the study's ECG, v3 reads 0.32 bpm off and v2 0.48 bpm on the 56 audit-accepted recordings both report. Neither model produced a VV-60 reading more than 10 bpm off. The window-level gain NB_P3_31 measured persists through the app's readout (like-for-like VV-60 window MAE 1.89 against 3.17 bpm; at Fitzpatrick 6, window MAE 6.78 against 10.53), but the recording-level median and the quality gates had already turned v2's windows into accurate readings, so the gain appears as a small improvement in accuracy rather than as additional readings.

**No guard set shows a regression.** On UBFC-Phys T1 both models report all 54 eligible recordings, with like-for-like window MAE 6.81 against 6.97 bpm. On v2's validation split, v3 reports 399 of 404 eligible recordings against 396, with like-for-like window MAE 4.59 against 4.60 bpm pooled and 4.93 against 4.90 on MCD (margins +0.5 and +0.35). On PhysDrive, v3 reports 106 of 263 recordings against v2's 87 with the same 4 wrong readings; window error stays near 21 bpm for both models, so neither is usable while driving.

**High motion is the exception.** On ECG-Fitness Part 0's primary set, 16 of v3's 34 reported readings are wrong against 14 of v2's 33: within the stress margin of +2, at its limit. The difference lies in the high-motion activities 05 and 06, where v3 reports 20 of 28 recordings against 16, and 178 of its 276 scored windows are confidently wrong against 93 of 171; NB_P3_29's hypothesis H1 (confident-wrong windows concentrate in 05/06) holds for v3 and remains unresolved for v2. Under moderate motion (02, 04), v3 reports 1 of 19 recordings against 3. The gates pass more of v3's high-motion readings, and too many of those are wrong.

**POS, for reference.** POS reports 11 of 23 recordings at types 5-6 and 136 of 404 on v2's validation split, against 20-21 and 396-399 for the two PhysNet models; 19 of its 33 reported ECG-Fitness readings are wrong.

**What the result is.** A decision about the app's default, taken on 23 recordings at Fitzpatrick 5-6 under a rule fixed before the runs. It is not evidence about skin tone in either direction: one gain against no loss gives McNemar's one-sided p = 0.50, and the subject-bootstrap interval for the coverage difference spans +0.0 to +13.0 points.

**Settled and open.** Settled: `hr_physnet_v2` remains `DEFAULT_ESTIMATOR`, and `hr_physnet_v3` stays registered as a candidate. Open: whether v3's accuracy gain justifies adoption is a different question from the one this battery asked. It requires its own rule, fixed in advance and tested on recordings that played no part in selecting v3 or in this battery (MMPD, or new captures). The ECG-Fitness result places the motion gate before any change of default.